# htcie — Step-by-Step Usage Guide

**htcie** (Heat Transfer Correlation Intelligence Engine) is a structured pipeline for evaluating convective heat transfer correlations.

Given a physical state — fluid properties, geometry, boundary conditions, and flow velocity — it answers five questions:

1. **Which correlations apply?** (applicability filters by Re, Pr, geometry, boundary type)
2. **What does each method predict?** (Nusselt number, heat transfer coefficient, uncertainty bands)
3. **Which method is most trustworthy?** (deterministic scoring across 8 factors)
4. **How confident should we be?** (inter-method spread + literature uncertainty)
5. **Why?** (human-readable explanation of every decision)

This notebook walks through each stage of the pipeline step by step.

In [11]:
import json
import plotly.graph_objects as go

from htcie.core.loader import build_registry
from htcie.core.pipeline import run_evaluation
from htcie.core.state import (
    EngineeringState,
    FluidProperties,
    Geometry,
    BoundaryConditions,
    FlowState,
)

registry = build_registry()
print(
    f"Loaded {len(registry.all())} correlations across {len(registry.families())} families"
)

Loaded 13 correlations across 3 families


## The Engineering State

`EngineeringState` is the canonical representation of the physical problem. It is a Pydantic model that holds:

- **`fluid`** — `FluidProperties`: density (kg/m³), dynamic viscosity (Pa·s), thermal conductivity (W/m·K), heat capacity (J/kg·K)
- **`geometry`** — `Geometry`: geometry type (`circular_tube`, `flat_plate`, `cylinder`), characteristic length (m), hydraulic diameter (m), optional roughness
- **`boundary`** — `BoundaryConditions`: boundary type (`constant_wall_temperature` or `constant_heat_flux`)
- **`flow`** — `FlowState`: velocity (m/s)

From these inputs, the engine derives two dimensionless groups:

- **Reynolds number** Re = ρ U D / μ — the ratio of inertial to viscous forces; governs whether flow is laminar (Re < 2300), transitional (2300–10000), or turbulent (Re > 10000)
- **Prandtl number** Pr = cp μ / k — the ratio of momentum diffusivity to thermal diffusivity; governs the relative thickness of velocity and thermal boundary layers

Keeping state separate from method implementations ensures that the same physical problem can be evaluated against any number of correlations without mutation.

In [12]:
with open("sample_internal_convection_input.json") as f:
    raw = json.load(f)

state = EngineeringState(**raw)
print(f"Re = {state.reynolds:.1f}")
print(f"Pr = {state.prandtl:.3f}")
print(f"Geometry: {state.geometry.geometry_type}")

Re = 22404.5
Pr = 6.200
Geometry: circular_tube


## The 7-Stage Pipeline

`run_evaluation(state, registry)` runs a deterministic seven-stage pipeline:

1. **Applicability** — each correlation declares its valid Re range, Pr range, geometry type, and required boundary condition. Correlations that fail any check are excluded with an explicit reason.
2. **Evaluate** — each applicable correlation computes the Nusselt number Nu from the dimensionless groups.
3. **Uncertainty** — each correlation declares a literature-derived uncertainty percentage. The engine computes `h = Nu · k / D` and propagates bounds as `h_low = h · (1 − δ)`, `h_high = h · (1 + δ)`.
4. **Rank** — correlations are scored on 8 deterministic factors (see below) and sorted best-first.
5. **Confidence** — the inter-method spread (stdev / mean of h values) and the winning score together determine a confidence class: `high`, `medium`, or `low`.
6. **Explain** — a structured explanation object is assembled: best method, score breakdown, exclusion reasons, key assumptions, extrapolation warnings.
7. **Report** — all outputs are packed into an `HtcieReport`, which is fully serializable via `.to_dict()`.

In [13]:
report = run_evaluation(state, registry)
if report is None:
    print("No applicable methods for the given state.")
else:
    print(f"Applicable: {report.applicable}")
    print(f"Excluded:   {[e['key'] for e in report.excluded]}")

Applicable: ['internal.dittus_boelter', 'internal.gnielinski', 'internal.petukhov', 'internal.sieder_tate']
Excluded:   ['external.churchill_bernstein', 'external.hilpert', 'external.pohlhausen_plate', 'external.turbulent_plate', 'external.zukauskas_cylinder', 'internal.churchill_ozoe', 'internal.shah_laminar', 'tube_banks.grimison', 'tube_banks.zukauskas']


## Applicability Checks

Every correlation in the registry declares its valid domain. Applicability is checked on three axes:

- **Geometry type** — Dittus-Boelter applies only to circular tubes, not flat plates or external flow
- **Reynolds number range** — e.g. turbulent correlations require Re > 10 000; laminar correlations require Re < 2300
- **Prandtl number range** — e.g. some correlations are invalid for liquid metals (very low Pr) or viscous oils (very high Pr)
- **Boundary condition** — some correlations are specific to constant wall temperature (T_w = const) or constant heat flux (q_w = const)

When a correlation fails a check, it is excluded with an **explicit reason string** — never silently dropped. This makes the audit trail complete.

In [14]:
for exc in report.excluded:
    print(f"  {exc['key']}: {exc['reason']}")

  external.churchill_bernstein: Geometry type 'circular_tube' does not match required 'cylinder_crossflow'.
  external.hilpert: Geometry type 'circular_tube' does not match required 'cylinder_crossflow'.
  external.pohlhausen_plate: Geometry type 'circular_tube' does not match required 'flat_plate'.
  external.turbulent_plate: Geometry type 'circular_tube' does not match required 'flat_plate'.
  external.zukauskas_cylinder: Geometry type 'circular_tube' does not match required 'cylinder_crossflow'.
  internal.churchill_ozoe: Reynolds number 2.24e+04 is above maximum 2300.
  internal.shah_laminar: Reynolds number 2.24e+04 is above maximum 2300.
  tube_banks.grimison: Geometry type 'circular_tube' does not match required 'tube_bank'.
  tube_banks.zukauskas: Geometry type 'circular_tube' does not match required 'tube_bank'.


## Evaluations and Uncertainty Bands

For each applicable correlation, the engine returns:

- **`value`** — the predicted Nusselt number Nu (dimensionless)
- **`h`** — heat transfer coefficient in W/m²·K, derived as `h = Nu · k / D`
- **`h_low` / `h_high`** — lower and upper bounds from literature uncertainty
- **`uncertainty_pct`** — the declared uncertainty percentage (e.g. 10% for Dittus-Boelter)
- **`uncertainty_note`** — qualitative context (e.g. 'smooth tubes only, not validated near transition')

The uncertainty bands represent the range within which the true value is expected to lie, based on the original authors' validation data. They do **not** include fluid property uncertainty or measurement error — those are separate concerns.

In [15]:
# Print table
header = f"{'Method':<35} {'Nu':>7} {'h':>9} {'h_low':>9} {'h_high':>9} {'±%':>6}"
print(header)
print("-" * len(header))
for e in report.evaluations:
    lo = f"{e['h_low']:.1f}" if e["h_low"] is not None else "N/A"
    hi = f"{e['h_high']:.1f}" if e["h_high"] is not None else "N/A"
    unc = f"{e['uncertainty_pct']:.1f}" if e["uncertainty_pct"] is not None else "N/A"
    print(f"{e['key']:<35} {e['value']:>7.2f} {e['h']:>9.1f} {lo:>9} {hi:>9} {unc:>6}")

# Error-bar chart of h +/- uncertainty
keys = [e["key"].split(".")[-1] for e in report.evaluations]
h_vals = [e["h"] for e in report.evaluations]
h_low = [e["h_low"] for e in report.evaluations]
h_high = [e["h_high"] for e in report.evaluations]

error_minus = [h - lo if lo else 0 for h, lo in zip(h_vals, h_low)]
error_plus = [hi - h if hi else 0 for h, hi in zip(h_vals, h_high)]

fig = go.Figure(
    go.Bar(
        y=keys,
        x=h_vals,
        orientation="h",
        error_x=dict(
            type="data", symmetric=False, array=error_plus, arrayminus=error_minus
        ),
        marker_color="#3498db",
    )
)
fig.update_layout(
    title="Heat Transfer Coefficient h with Uncertainty Bands",
    xaxis_title="h [W/m\u00b2\u00b7K]",
    height=350,
)
fig.show()

Method                                   Nu         h     h_low    h_high     ±%
--------------------------------------------------------------------------------
internal.dittus_boelter              144.20    8651.9    6489.0   10814.9   25.0
internal.gnielinski                  156.12    9367.5    8430.7   10304.2   10.0
internal.petukhov                    159.29    9557.3    8601.6   10513.1   10.0
internal.sieder_tate                 149.89    8993.3    7194.7   10792.0   20.0


## Scoring and Ranking

htcie ranks applicable correlations using **8 deterministic scoring factors**:

1. **Validity fit** — how close the operating Re and Pr are to the center of the correlation's valid range (penalises operation near the boundary)
2. **Geometry match** — does the correlation geometry exactly match, or is it an approximation?
3. **Regime match** — laminar, transitional, or turbulent — does the correlation target the actual flow regime?
4. **Boundary condition match** — constant wall temperature vs. constant heat flux
5. **Correction completeness** — does the correlation include viscosity correction, entrance length, or other physically important factors?
6. **Pedigree** — how well-validated is the correlation in the literature (number of data points, range of conditions tested)?
7. **Literature uncertainty** — correlations with lower declared uncertainty score higher
8. **Extrapolation penalty** — if any input is outside the stated valid range, a penalty is applied

Every score is a deterministic function of the state and the correlation metadata — no hidden heuristics. If the ranking changes, it is because the metadata or logic changed, not because of random weighting.

In [16]:
# Print ranking table
print(f"{'Rank':<5} {'Method':<35} {'Score':>7}  Breakdown")
print("-" * 80)
for i, r in enumerate(report.ranking, 1):
    breakdown_str = ", ".join(f"{k}={v:.2f}" for k, v in r["breakdown"].items())
    print(f"{i:<5} {r['key']:<35} {r['score']:>7.3f}  {breakdown_str}")

# Parallel coordinates plot — each line is a method, each axis is a score factor
if report.ranking:
    factors = list(report.ranking[0]["breakdown"].keys())
    scores = [r["score"] for r in report.ranking]
    method_names = [r["key"].split(".")[-1] for r in report.ranking]
    n = len(report.ranking)

    # First axis: correlation name (numeric index with named ticks)
    name_dim = dict(
        label="correlation",
        values=list(range(n)),
        range=[-0.5, n - 0.5],
        tickvals=list(range(n)),
        ticktext=method_names,
    )
    factor_dims = [
        dict(
            label=factor.replace("_", " "),
            values=[r["breakdown"].get(factor, 0) for r in report.ranking],
            range=[0.0, 1.0],
        )
        for factor in factors
    ]
    score_dim = dict(
        label="total score",
        values=scores,
        range=[min(scores) * 0.95, max(scores) * 1.05],
    )

    fig = go.Figure(
        go.Parcoords(
            line=dict(
                color=scores,
                colorscale="plasma",
                showscale=True,
                colorbar=dict(title="Score", thickness=12),
                cmin=min(scores) * 0.95,
                cmax=max(scores) * 1.05,
            ),
            dimensions=[name_dim] + factor_dims + [score_dim],
            labelfont=dict(size=10),
            tickfont=dict(size=9),
            rangefont=dict(size=8),
        )
    )
    fig.update_layout(
        title=dict(text="Ranking Score Breakdown by Factor", y=0.97, yanchor="top"),
        height=300,
        margin=dict(l=80, r=80, t=80, b=20),
    )
    fig.show()

Rank  Method                                Score  Breakdown
--------------------------------------------------------------------------------
1     internal.gnielinski                   0.652  validity_fit=0.01, geometry_match=1.00, regime_match=1.00, boundary_match=1.00, correction_score=0.50, pedigree=1.00, uncertainty_score=1.00, extrapolation_penalty=0.00
2     internal.petukhov                     0.651  validity_fit=0.00, geometry_match=1.00, regime_match=1.00, boundary_match=1.00, correction_score=0.50, pedigree=1.00, uncertainty_score=1.00, extrapolation_penalty=0.00
3     internal.dittus_boelter               0.628  validity_fit=0.23, geometry_match=1.00, regime_match=1.00, boundary_match=1.00, correction_score=0.50, pedigree=0.50, uncertainty_score=0.40, extrapolation_penalty=0.00
4     internal.sieder_tate                  0.613  validity_fit=0.03, geometry_match=1.00, regime_match=1.00, boundary_match=1.00, correction_score=0.50, pedigree=0.50, uncertainty_score=0.70, extra

## The Explanation Object

The `report.explanation` object captures the reasoning behind the result in structured form:

- **`best_method`** — key of the top-ranked correlation
- **`best_score`** — its composite score
- **`score_breakdown`** — per-factor scores for the best method
- **`why_others_excluded`** — map of excluded keys to reasons
- **`key_assumptions`** — physical assumptions made by the best method
- **`confidence_class`** — `high`, `medium`, or `low`
- **`extrapolation_warnings`** — any inputs outside the validated range
- **`recommendation_note`** — a plain-language summary

Calling `.to_text()` renders all of this into a human-readable report you can include in a design document or engineering notebook.

In [17]:
print(report.explanation.to_text())
print(f"\nConfidence class: {report.confidence}")
print(f"Spread: {report.spread['relative_spread']:.1%}")

Recommended method: internal.gnielinski
Score: 0.652

Score breakdown:
  validity_fit: 0.008
  geometry_match: 1.000
  regime_match: 1.000
  boundary_match: 1.000
  correction_score: 0.500
  pedigree: 1.000
  uncertainty_score: 1.000
  extrapolation_penalty: 0.000

Confidence: high
Key assumptions:
  - Smooth circular tube, fully developed or thermally developing turbulent flow
  - Fluid properties at bulk mean temperature
  - Petukhov friction factor used: $f = (0.790 \ln Re - 1.64)^{-2}$
Excluded methods:
  external.churchill_bernstein: Geometry type 'circular_tube' does not match required 'cylinder_crossflow'.
  external.hilpert: Geometry type 'circular_tube' does not match required 'cylinder_crossflow'.
  external.pohlhausen_plate: Geometry type 'circular_tube' does not match required 'flat_plate'.
  external.turbulent_plate: Geometry type 'circular_tube' does not match required 'flat_plate'.
  external.zukauskas_cylinder: Geometry type 'circular_tube' does not match required 'cyli

## Building a State Inline and Sweeping a Parameter

You don't need to load from JSON. States can be built inline, which makes it easy to sweep a parameter — for example, to understand how Nu changes with velocity across the turbulent regime.

In [18]:
import numpy as np

velocities = list(np.linspace(0.5, 5.0, 10))
sweep_results = []

for U in velocities:
    s = EngineeringState(
        fluid=FluidProperties(
            density=997.0,
            viscosity=0.00089,
            thermal_conductivity=0.6,
            heat_capacity=4180.0,
        ),
        geometry=Geometry(
            geometry_type="circular_tube",
            characteristic_length=0.01,
            hydraulic_diameter=0.01,
        ),
        boundary=BoundaryConditions(boundary_type="constant_wall_temperature"),
        flow=FlowState(velocity=U),
    )
    r = run_evaluation(s, registry)
    if r and r.ranking:
        best_key = r.ranking[0]["key"]
        best_eval = next(e for e in r.evaluations if e["key"] == best_key)
        sweep_results.append(
            {
                "velocity": U,
                "Re": s.reynolds,
                "Nu": best_eval["value"],
                "h": best_eval["h"],
                "h_low": best_eval["h_low"] or best_eval["h"],
                "h_high": best_eval["h_high"] or best_eval["h"],
                "method": best_key.split(".")[-1],
            }
        )
        print(
            f"U={U:.2f} m/s  Re={s.reynolds:.0f}  best={best_key.split('.')[-1]}  Nu={best_eval['value']:.1f}  h={best_eval['h']:.1f}"
        )

# Nu vs Velocity
fig_nu = go.Figure(
    go.Scatter(
        x=[r["velocity"] for r in sweep_results],
        y=[r["Nu"] for r in sweep_results],
        mode="lines+markers",
        marker=dict(size=10, color="#e74c3c"),
        line=dict(color="#e74c3c"),
    )
)
fig_nu.update_layout(
    title="Nu vs Velocity (best method at each point)",
    xaxis_title="Velocity [m/s]",
    yaxis_title="Nusselt number Nu",
    height=320,
)
fig_nu.show()

# HTC vs Velocity with uncertainty bands and correlation labels
vs = [r["velocity"] for r in sweep_results]
hs = [r["h"] for r in sweep_results]
lows = [r["h_low"] for r in sweep_results]
his = [r["h_high"] for r in sweep_results]

fig_h = go.Figure()
fig_h.add_trace(
    go.Scatter(
        x=vs + vs[::-1],
        y=his + lows[::-1],
        fill="toself",
        fillcolor="rgba(52, 152, 219, 0.2)",
        line=dict(color="rgba(0,0,0,0)"),
        name="Uncertainty band",
        showlegend=True,
    )
)
fig_h.add_trace(
    go.Scatter(
        x=vs,
        y=hs,
        mode="lines+markers",
        marker=dict(size=10, color="#3498db"),
        line=dict(color="#3498db"),
        name="h (best method)",
    )
)
for r in sweep_results:
    fig_h.add_annotation(
        x=r["velocity"],
        y=r["h_low"],
        text=r["method"],
        showarrow=False,
        yanchor="top",
        yshift=-4,
        font=dict(size=8),
        textangle=-45,
    )
fig_h.update_layout(
    title="Heat Transfer Coefficient h vs Velocity with Uncertainty Bands",
    xaxis_title="Velocity [m/s]",
    yaxis_title="h [W/m\u00b2\u00b7K]",
    height=400,
    margin=dict(b=80),
)
fig_h.show()

U=0.50 m/s  Re=5601  best=gnielinski  Nu=43.5  h=2608.2
U=1.00 m/s  Re=11202  best=gnielinski  Nu=84.3  h=5055.4
U=1.50 m/s  Re=16803  best=gnielinski  Nu=121.2  h=7273.8
U=2.00 m/s  Re=22404  best=gnielinski  Nu=156.1  h=9367.5
U=2.50 m/s  Re=28006  best=dittus_boelter  Nu=172.4  h=10342.9
U=3.00 m/s  Re=33607  best=dittus_boelter  Nu=199.5  h=11967.0
U=3.50 m/s  Re=39208  best=dittus_boelter  Nu=225.6  h=13537.7
U=4.00 m/s  Re=44809  best=dittus_boelter  Nu=251.1  h=15063.9
U=4.50 m/s  Re=50410  best=dittus_boelter  Nu=275.9  h=16552.3
U=5.00 m/s  Re=56011  best=dittus_boelter  Nu=300.1  h=18008.0


In [19]:
import json

d = report.to_dict()
print(json.dumps(d, indent=2, default=str)[:1000], "...")

{
  "engine_version": "0.1.0",
  "timestamp": "2026-03-29, 16:07:32 UTC",
  "input_state": {
    "fluid": {
      "density": 997.0,
      "viscosity": 0.00089,
      "thermal_conductivity": 0.6,
      "heat_capacity": 4180.0,
      "wall_viscosity": null
    },
    "geometry": {
      "geometry_type": "circular_tube",
      "characteristic_length": 0.01,
      "hydraulic_diameter": 0.01,
      "roughness": 0.0,
      "pitch_transverse": null,
      "pitch_longitudinal": null,
      "arrangement": null
    },
    "boundary": {
      "boundary_type": "constant_wall_temperature",
      "wall_temperature": null,
      "bulk_temperature": null
    },
    "flow": {
      "velocity": 2.0,
      "mass_flow_rate": null,
      "developing_length": null
    },
    "reynolds": 22404.494382022476,
    "prandtl": 6.200333333333333,
    "relative_roughness": 0.0,
    "graetz": null,
    "entry_length_ratio": null,
    "pitch_ratio_transverse": null,
    "pitch_ratio_longitudinal": null
  },
  "derive

## Using the Report Downstream

`report.to_dict()` returns a fully JSON-serializable dictionary. Common uses:

- **API response** — pass it directly as the body of a FastAPI endpoint response
- **Persist to file** — `json.dump(report.to_dict(), open('result.json', 'w'))` for archival or audit
- **Test assertions** — assert on `report.confidence`, `report.ranking[0]['key']`, or specific Nu values
- **Report generation** — feed into a Jinja2 template or PDF renderer for design documentation

Because the report is deterministic — same state, same registry, same result — it can be stored alongside the design file and reproduced exactly at any future point.

In [20]:
from htcie.reports.renderer import save_html

save_html(report, "htcie_report.html")
print(
    "Report written to htcie_report.html — open in any browser, or use File > Print > Save as PDF."
)

Report written to htcie_report.html — open in any browser, or use File > Print > Save as PDF.
